# 波形オートエンコーダーの比較

`kickoff_notebook.ipynb` の1D CNN AEを参考に、入力表現を比較します。
全PV共通で1モデル／比較手法とし、PV名（PMData／DODData統合）とbeam_modeを条件入力にします。
元波形AEではサンプルの間引き・補間をせず、入力と出力の長さを一致させます。
異なる長さはバッチだけ分け、モデルの重みは共通です。内部の圧縮と復号はAEの一部です。

学習対象は除外後のデータセットです。除外49サイクルは評価だけに使います。
残ったデータは「正常候補」であり、正常が確認されたラベルではありません。

## 診断機のゴールと追加の異常データ

ゴールは `kickoff_notebook.ipynb` と同じ、正常波形の再構成を学習し、復元誤差を異常スコアとして異常波形を検出することです。全PV共通のAEに波形・PV名・beam_modeを入力します。元波形、間引き、区間集約、PCA＋AEの比較も、この同じゴールで行います。

波形ごとのスコアと閾値の判定を残した上で、同じcycle_idの中で1つでもPVが閾値を超えれば、そのサイクルを異常候補とします。サイクルのスコアはPVスコアの最大値です。原因候補のPV名・スコアも出力します。

追加の異常データについて分かるのは「cycle_idのどこかのPVに異常がある」というサイクルラベルです。そのサイクルの全PVに異常ラベルを付けません。異常サイクルは正常学習・正常のモデル選択・正常の閾値校正から除外し、サイクル単位の検証・テストに使います。同じ異常イベントの連続サイクルや複数PVを、調整用と最終評価用に分散させません。

既知のHBLの異常2波形は同じcycle_idなので、サイクルの検出率を評価する際は1件です。波形の異常PVが不明な追加データでは、PVのランキングは原因候補であり確定した故障PVではありません。

中央値＋5σの基準はAEの波形復元誤差に適用します。サイクル判定はその判定のORであり、別の分類器へ置き換えません。複数PVを調べるため、モデル比較では波形の候補率に加えてサイクルの候補率を確認します。

## W&Bの公式採点と入力形式（配布リポジトリで確認）

公式 `score.py` のデータは `data/sample/ess_llrf_sample.parquet` です。210イベントの `Q` 列をfloat32へ変換し、学習168件だけの平均・標準偏差で正規化します。モデルへの入力と復元出力は `(N, 1, 256)` です。PVとbeam_modeはメタデータにありますが、評価関数は `model(X)` の形で波形だけを渡すため、条件付きAEには評価用アダプターが必要です。

`data/sample/holdout_ids.json` は168学習行と42評価行の固定分割です。IDは210行のサンプル内の行番号で、cycle_idではありません。採点は42件の正規化後の復元MSEで、W&Bの `anomaly/holdout_mse` に記録します。異常検出率やサイクル判定の精度とは別の指標です。PCAなどの内部表現を使う場合も、採点用には256点の復元波形を返します。

この比較コードの独自分割は公式分割と異なり、保存済み20epochのチェックポイントは公式holdoutのうち23サイクルを学習側に含みます。そのチェックポイントを公式採点に流用しません。公式評価に対応する前に、固定holdoutサイクルを学習・モデル選択・校正から除外し、入力の前処理と条件入力のアダプターを整え、新しい出力先で学習する必要があります。

公式参照ファイルは `official_anomaly_reference/` にコミットを固定して保存しています。形式と分割の監査は `official_anomaly_reference/input_and_scoring_audit.json`、過去の分割との重複は `official_anomaly_reference/official_holdout_vs_previous_split.csv` です。公式スコアの計算・W&Bへの送信・学習再開はまだ行っていません。

参照: [公式score.py](https://github.com/aghribi/artifact-hackathon-2026/blob/32f2870ea23a3342e7dec390da120267b1f68c0c/challenges/anomaly-detection/score.py)、[採点仕様](https://github.com/aghribi/artifact-hackathon-2026/blob/32f2870ea23a3342e7dec390da120267b1f68c0c/GRADING.md)。

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
OUTPUT = ROOT / "waveform_autoencoder_comparison"
RUN_TRAINING = False  # 新規学習／チェックポイントからの再開をするときだけTrue

ANOMALY_CYCLES_CSV = None  # 確認済み異常のcycle_id列を持つCSV。追加データ入手後に指定。


比較対象は、元波形、4点ごとの単純間引き、4点・16点の区間平均、16点の最小・最大値、およびPCAの32・64・128主成分を入力したAEです。PCA単体の復元誤差も3通り計算します。

PCAはAEの学習用サイクルだけでfitします。PCA＋AEの判定スコアには、PCAで失われた成分も含む元波形に対する復元誤差を使います。min/maxだけは復元した上下包絡の外側への距離を使うため、他のMSEと値を直接比較しません。

同一収録アーカイブと同一サイクルは分割をまたぎません。運転モードごとにアーカイブを学習・モデル選択・閾値校正・テストへ割り当てます。標準化係数とPCAは学習集合だけから計算します。波形単位の標準化はしません。

In [ ]:
if RUN_TRAINING:
    import subprocess, sys
    command = [
        sys.executable, "-u", str(ROOT / "compare_waveform_autoencoders.py"),
        "--output", str(OUTPUT), "--epochs", "30", "--patience", "5",
    ]
    if ANOMALY_CYCLES_CSV is not None:
        command += ["--anomaly-cycles", str(ANOMALY_CYCLES_CSV)]
    subprocess.run(command, check=True)
else:
    print("学習は停止中です。ここでは設定と保存済みデータだけを確認します。")


In [ ]:
config = json.loads((OUTPUT / "configuration.json").read_text())
print("分割ごとの波形数:", config["counts"])
print("標準化:", config["normalization"])
print("閾値:", config["threshold"])
splits = pd.read_csv(OUTPUT / "sample_splits.csv")
display(splits.groupby(["split", "beam_mode"]).agg(
    waveforms=("original_row", "size"), cycles=("cycle_id", "nunique")))
assert splits.groupby("cycle_id")["split"].nunique().max() == 1


閾値は学習に使っていない校正集合の復元誤差から **中央値＋5σ** で決定します。σは標準偏差（ddof=0）です。テスト集合の候補率は、正常ラベルがないため確定した誤検知率ではありません。

確認済み異常はHBL-010RFCのDwn0-Cmp1とDwn6-Cmp1、cycle_id=16122596526の2波形です。この2例をどれだけ検出したかと、未確認テスト集合の候補率、計算時間を併記します。

In [ ]:
comparison_path = OUTPUT / "comparison.csv"
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
if comparison.empty:
    print("比較結果はまだありません。学習は停止中です。")
else:
    columns = ["method", "epochs", "known_detected", "known_total", "known_cycles_detected",
               "known_cycles_total", "test_candidate_cycles", "test_cycles", "test_cycle_candidate_rate",
               "test_candidates", "test_waveforms", "training_seconds"]
    display(comparison[[c for c in columns if c in comparison]])
    from IPython.display import Image, display
    display(Image(filename=str(OUTPUT / "comparison.png")))


In [ ]:
known_results_path = OUTPUT / "known_anomaly_results.csv"
known_results = pd.read_csv(known_results_path) if known_results_path.exists() else pd.DataFrame()
if not known_results.empty:
    display(known_results)
else:
    print("既知波形の比較結果は学習後に表示します。")


In [ ]:
if comparison.empty:
    print("元波形と復元波形の比較図は学習後に表示します。")
else:
    # 元波形と復元波形の比較。表示の間引きだけ行い、学習・スコアは全サンプルです。
    from waveform_anomaly_comparison import RawCache
    known = pd.read_csv(ROOT / "waveform_anomaly_comparison" / "known_anomalies.csv")
    cache = RawCache(ROOT / "anomaly-pca-reuse-20261015")
    original = cache.read(known.original_row.astype(int).tolist())
    methods = comparison.method.tolist()
    fig, axes = plt.subplots(len(known), len(methods), figsize=(4 * len(methods), 7), squeeze=False)
    for col, method in enumerate(methods):
        with np.load(OUTPUT / (method + "_known_reconstructions.npz")) as saved:
            for row, item in enumerate(known.itertuples()):
                ax = axes[row, col]
                wave = original[row]
                step = max(1, int(np.ceil(len(wave) / 3000)))
                x = np.arange(0, len(wave), step)
                ax.plot(x, wave[::step], label="Original", lw=.9)
                recon = saved[str(item.original_row)]
                if recon.shape[0] == 2:
                    ax.fill_between(x, np.minimum(recon[0],recon[1])[::step],
                                    np.maximum(recon[0],recon[1])[::step], color="tab:orange",alpha=.4,label="Reconstructed bounds")
                else:
                    ax.plot(x, recon[0,::step],label="Reconstructed",lw=.8,alpha=.8)
                verdict = known_results.loc[(known_results.method==method) &
                                            (known_results.original_row==item.original_row),"detected"].iloc[0]
                ax.set_title(f"{method} | detected={verdict}\n{item.pv.split(':')[-1]}",fontsize=9)
                ax.set_xlabel("Original sample index")
                ax.grid(alpha=.2)
                if col==0: ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()


In [ ]:
# 同一サイクル内の1つ以上のPVが基準を超えたら、サイクルを異常候補にする。
SELECTED_METHOD = "raw"
cycle_path = OUTPUT / (SELECTED_METHOD + "_cycle_predictions.csv")
pv_path = OUTPUT / (SELECTED_METHOD + "_candidate_pvs.csv")
if cycle_path.exists():
    cycle_predictions = pd.read_csv(cycle_path)
    display(cycle_predictions.loc[cycle_predictions.status.eq("anomaly_candidate")])
    display(pd.read_csv(pv_path))
else:
    print("サイクル判定と原因PV候補は、学習・スコア計算後に表示します。")


学習曲線が30epochの上限まで改善を続けた場合、収束した比較とは見なさず追加学習で確認します。PCA単体は波形のみの参考基準です。AEはすべてPV名とbeam_modeを使います。2波形だけで全異常に対する検出率を推定せず、候補を目視確認して正常・異常ラベルを増やして評価します。

参考: [PyTorch Conv1d](https://docs.pytorch.org/docs/2.10/generated/torch.nn.Conv1d.html)、[Kerasの時系列AE例](https://keras.io/examples/timeseries/timeseries_anomaly_detection/)。